# Notebook 9: Four Ways to Do SBI, Side by Side

We have now solved inference problems in four ways: with our own code (Notebook 4), with `sbi` (Notebook 5), with `ltu-ili` (Notebook 6) and with `sbikit` (Notebook 8). This last notebook puts them next to each other on one problem, the Gaussian toy model, with the same training simulations, the same observation and the same exact posterior to compare with.

Each section contains the complete code for one tool in a single cell, so you can use this notebook as a reference to look up how the same task is written in each of them.

**What you will learn**
- how the same analysis looks in the four tools
- how their results, their runtime and the amount of code compare
- which tool fits which situation

**Runtime:** about three minutes on a laptop.

In [ ]:
import time
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt

# The toy problem from Notebook 4
from tutorial_utils import (gaussian_benchmark, GAUSSIAN_PRIOR_LOW, GAUSSIAN_PRIOR_HIGH,
                            analytic_moments, plot_posterior_2d)

torch.manual_seed(42)
np.random.seed(42)

# Our models are small, so they do not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## The Task

- **Training:** 10000 simulations of the Gaussian toy model (parameters: mean and log standard deviation, data: ten draws).
- **Result 1:** 5000 posterior samples for the observation `x_obs`.
- **Result 2:** posterior samples for 100 test simulations, which we compare with the exact posterior of each (width and position).

All tools use a neural spline flow of the same size where the tool lets us choose (64 hidden units, 4 transforms). Our own code uses the affine coupling flow from Notebook 3.

In [ ]:
benchmark = gaussian_benchmark()
theta_train, x_train = benchmark.theta_train, benchmark.x_train
theta_true, x_obs = benchmark.theta_obs, benchmark.x_obs

num_eval = 100
x_eval = benchmark.x_test[:num_eval]

results = {}

def store_result(name, samples_obs, samples_eval, train_time, sample_time):
    """samples_obs: [5000, 2] for the observation, samples_eval: [num_eval, n, 2] for the test simulations"""
    results[name] = dict(samples_obs=np.asarray(samples_obs), samples_eval=np.asarray(samples_eval),
                         train_time=train_time, sample_time=sample_time)
    print(f'\n{name}: training {train_time:.0f} s, sampling the observation {sample_time:.2f} s')

## 1. From Scratch (NPE)

The conditional flow from Notebook 3 and a training loop. This is a compact version of Notebook 4 (without the validation set).

In [ ]:
from flow_utils import ConditionalRealNVP

# Normalization of parameters and data
theta_mean, theta_std = theta_train.mean(dim=0), theta_train.std(dim=0)
x_mean, x_std = x_train.mean(), x_train.std()
theta_norm, x_norm = (theta_train - theta_mean) / theta_std, (x_train - x_mean) / x_std

# Training
start = time.time()
flow = ConditionalRealNVP(input_dim=2, context_dim=10, hidden_dim=64, n_layers=8, max_log_scale=3.0)
optimizer = torch.optim.Adam(flow.parameters(), lr=2e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
for epoch in range(150):
    for batch in torch.randperm(len(theta_norm)).split(250):
        optimizer.zero_grad()
        loss = -flow.log_prob(theta_norm[batch], x_norm[batch]).mean()
        loss.backward()
        optimizer.step()
    scheduler.step()
train_time = time.time() - start

# Sampling
def sample_scratch(x, num_samples):
    # x: [B, 10] -> samples [B, num_samples, 2]
    context = ((x - x_mean) / x_std).repeat_interleave(num_samples, dim=0)
    with torch.no_grad():
        samples = flow.sample(context) * theta_std + theta_mean
    return samples.reshape(len(x), num_samples, 2)

start = time.time()
samples_obs = sample_scratch(x_obs, 5000)[0]
sample_time = time.time() - start

store_result('From scratch (NPE)', samples_obs, sample_scratch(x_eval, 1000), train_time, sample_time)

## 2. `sbi` (NPE)

In [ ]:
from sbi.inference import NPE
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform

prior = BoxUniform(low=GAUSSIAN_PRIOR_LOW, high=GAUSSIAN_PRIOR_HIGH)

# Training
start = time.time()
inference = NPE(prior=prior, density_estimator=posterior_nn(model='nsf', hidden_features=64, num_transforms=4))
density_estimator = inference.append_simulations(theta_train, x_train).train()
posterior = inference.build_posterior(density_estimator)
train_time = time.time() - start

# Sampling
start = time.time()
samples_obs = posterior.sample((5000,), x=x_obs, show_progress_bars=False)
sample_time = time.time() - start

samples_eval = posterior.sample_batched((1000,), x=x_eval, show_progress_bars=False)       # [1000, num_eval, 2]
store_result('sbi (NPE)', samples_obs, samples_eval.permute(1, 0, 2), train_time, sample_time)

## 3. `ltu-ili` (NPE)

To keep the runtime short, the "ensemble" consists of a single network here. In practice you would use several (Notebook 6).

In [ ]:
import ili
from ili.dataloaders import NumpyLoader
from ili.inference import InferenceRunner

logging.getLogger().setLevel(logging.WARNING)     # ili switches on INFO messages when it is imported

loader = NumpyLoader(x=x_train.numpy(), theta=theta_train.numpy())
prior_ili = ili.utils.Uniform(low=[-5.0, -2.0], high=[5.0, 1.0], device='cpu')

# Training
start = time.time()
runner = InferenceRunner.load(
    backend='sbi', engine='NPE', prior=prior_ili,
    nets=[ili.utils.load_nde_sbi(engine='NPE', model='nsf', hidden_features=64, num_transforms=4)],
    train_args={'training_batch_size': 200, 'learning_rate': 5e-4, 'stop_after_epochs': 20},
)
posterior_ili, summaries = runner(loader=loader, seed=1)
train_time = time.time() - start

# Sampling
start = time.time()
samples_obs = posterior_ili.sample((5000,), x_obs, show_progress_bars=False)
sample_time = time.time() - start

samples_eval = torch.stack([posterior_ili.sample((1000,), x, show_progress_bars=False) for x in x_eval])
store_result('ltu-ili (NPE)', samples_obs, samples_eval, train_time, sample_time)

## 4. `sbikit` (NLE)

`sbikit` implements neural likelihood estimation. The data vector has only ten entries, so we skip the compression step and learn the likelihood of the data directly. The posteriors of the observation and of all test simulations are then sampled in one batched MCMC run.

In [ ]:
import sbikit as sk

prior_sk = sk.Prior(GAUSSIAN_PRIOR_LOW, GAUSSIAN_PRIOR_HIGH, names=['mean', 'log_std'])

# Training (9000 simulations for training, 1000 for validation)
start = time.time()
torch.manual_seed(0)
nle = sk.NLE(n_x=10, n_theta=2, flow={'model': 'nsf', 'hidden_features': 64, 'num_transforms': 4})
nle.fit(x_train[:9000], theta_train[:9000], x_train[9000:], theta_train[9000:], seed=0)
train_time = time.time() - start

# Sampling: the observation ...
start = time.time()
config = sk.MCMCConfig(nwalkers=64, init='prior', burn_in=500, block=500, max_steps=4000, n_samples=5000, seed=0)
posterior_sk = sk.sample_posteriors(nle, x_obs, prior_sk, config)
sample_time = time.time() - start

# ... and all test simulations in one batched run
config_eval = sk.MCMCConfig(nwalkers=32, init='prior', burn_in=300, block=300, max_steps=1500,
                            rhat=1.05, min_ess=400, n_samples=1000, seed=0)
posterior_sk_eval = sk.sample_posteriors(nle, x_eval, prior_sk, config_eval)
print(f'Converged: {posterior_sk_eval.converged.sum()} of {num_eval} test posteriors')

store_result('sbikit (NLE)', posterior_sk.samples[0], posterior_sk_eval.samples, train_time, sample_time)

## Comparison of the Results

First the posterior of the observation, compared with the exact posterior.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(21, 4.5))
for ax, (name, result) in zip(axs, results.items()):
    plot_posterior_2d(ax, result['samples_obs'], x_obs, theta_true, title=name)
axs[0].legend(loc='upper left')
plt.show()

One observation can be lucky or unlucky. For the 100 test simulations we therefore compare the standard deviation and the mean of each posterior with the exact values (as in Notebook 5): a width ratio of one and a bias of zero would be perfect. The table shows the median over the test simulations.

In [ ]:
exact = [analytic_moments(x) for x in x_eval]
exact_mean = torch.stack([moments[0] for moments in exact]).numpy()
exact_std = torch.stack([moments[1] for moments in exact]).numpy()

print(f"{'':20s} | {'width / exact width':^21s} | {'|bias| / exact width':^21s} | {'training':>8s} | {'sampling x_obs':>14s}")
print(f"{'':20s} | {'Mean':>9s} {'Log Std':>11s} | {'Mean':>9s} {'Log Std':>11s} | {'':>8s} | {'':>14s}")
for name, result in results.items():
    samples = result['samples_eval']                                   # [num_eval, n, 2]
    width_ratio = np.median(samples.std(axis=1) / exact_std, axis=0)
    bias = np.median(np.abs(samples.mean(axis=1) - exact_mean) / exact_std, axis=0)
    print(f"{name:20s} | {width_ratio[0]:9.2f} {width_ratio[1]:11.2f} | {bias[0]:9.2f} {bias[1]:11.2f} | "
          f"{result['train_time']:6.0f} s | {result['sample_time']:12.2f} s")

What to take from this comparison (the exact numbers change from run to run):

- **All four give the right answer.** With 10000 simulations, every tool recovers the exact posterior to within a few tenths of its width. The methods are the same, the packages differ in what they do for you.
- **NLE is the most accurate here**, because the likelihood of this model (ten independent Gaussian numbers) is easier to learn than the posterior. This is specific to the problem. With a large data vector, the likelihood becomes the harder object, and one needs the compression step of Notebooks 7 and 8.
- **NPE samples in milliseconds, NLE needs an MCMC.** If you need the posterior of one observation, this hardly matters. If you need thousands, it does, unless the sampler is batched.
- **Our own code is competitive on this simple problem**, but it has no early stopping, no diagnostics, and a flow which would not be flexible enough for harder posteriors.

## Which Tool for Which Job?

| | From scratch | `sbi` | `ltu-ili` | `sbikit` |
|---|---|---|---|---|
| Methods | whatever you write | NPE, NLE, NRE, flow matching, diffusion, sequential versions | NPE, NLE, NRE and sequential versions (through `sbi`), NPE (through `lampe`) | NLE |
| Compression of large data vectors | by hand | embedding network trained together with NPE | embedding network | dedicated compressors (VMIM, MOPED, IMNN, ...) |
| Ensembles of networks | by hand | by hand | built in, the default | `sk.Ensemble` |
| Sampling | by hand | direct (NPE), MCMC or variational inference (NLE), one observation at a time for MCMC | as `sbi`, plus `emcee` | batched ensemble MCMC for many observations, with convergence checks |
| Validation | by hand | `sbi.diagnostics`: SBC, TARP and more | `PosteriorCoverage`: a standard set of plots | `evaluate`: TARP with error bars, marginal coverage, figures of merit |
| Reproducibility | your scripts | `pickle` | configuration files for the whole analysis | saved models and run records |
| Lines of code for this task | about 25 | about 8 | about 10 | about 8 |

Some guidance:

- **You want to understand a method, or try a new idea:** write it yourself. It is less code than one might think, and you can test every piece against a problem with a known answer, as we did throughout this tutorial.
- **General-purpose inference, the widest choice of methods, amortized NPE:** `sbi`. It is the reference implementation which the other two packages build on.
- **A standardized, reproducible NPE analysis with ensembles and the usual validation plots,** in particular within a collaboration which shares configuration files: `ltu-ili`.
- **A large data vector, simulations which do not follow your prior, and posteriors for hundreds of test simulations:** compression plus NLE with a batched sampler, which is what `sbikit` is built for.

The tools can be combined: all of them work on the same arrays of simulations, and a coverage test or a comparison with a second method is cheap insurance for any result.

## The End

This is the end of the tutorial. From a single neuron in NumPy to calibrated posteriors from compressed data vectors, every step was built on the previous one:

1. a network is a chain of differentiable layers (Notebooks 1 and 2),
2. a network can represent a probability distribution, and a normalizing flow a very flexible one (Notebook 3),
3. trained on simulations, a conditional flow becomes a posterior or a likelihood (Notebook 4),
4. packages provide this in a robust form (Notebooks 5, 6 and 8),
5. and no result should be trusted before it is validated (all of them).

The best next step is to try these tools on a simulator of your own.